# FADE Quickstart

**Frequency-Adaptive Decay Encoding** — drop-in KV cache compression for HuggingFace transformers. **3.5–12× compression** with near-baseline quality (up to 23× in aggressive mode).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omodaka9375/fade/blob/main/examples/quickstart.ipynb)

In [ ]:
!pip install -q fade-kv

In [ ]:
import torch
from transformers import DynamicCache

from fade import FadeConfig, create_tiered_cache
from fade.backends import get_backend
from fade.eval.memory import cache_storage_bytes
from fade.patch import forward_with_tracking, load_model
from fade.policy import reassign_tiers_by_position

try:
    from fade.policy import reassign_tiers_adaptive
except ImportError:
    reassign_tiers_adaptive = None  # Not in PyPI yet, use position fallback
from fade.tracker import AttentionTracker

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
PROMPT = (
    "Explain CPU caches, Roman roads, photosynthesis, "
    "quantum computing, immune system, plate tectonics, "
    "jazz music, and KV caching. One paragraph each.\n"
)
MAX_NEW = 128
REASSIGN_EVERY = 32

print(f"Device: {DEVICE}")
model, tokenizer = load_model(MODEL_ID, device_map=DEVICE, dtype=DTYPE, attn_impl="sdpa")
num_layers = model.config.num_hidden_layers
head_dim = model.config.hidden_size // model.config.num_attention_heads
print(f"Model loaded: {num_layers} layers, head_dim={head_dim}")

## Drop-in with model.generate()

The simplest way to use FADE — just pass the cache to `model.generate()`.
Works with greedy, sampling, and beam search.

In [ ]:
# Drop-in: just pass the cache to model.generate()
cache = create_tiered_cache(model, config=FadeConfig.safe())
enc = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
out = model.generate(**enc, past_key_values=cache, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True)[:200])
print(f"\nKV cache: {cache.compressed_storage_bytes() / 1024:.0f} KB")

## Compression comparison

Compare all presets side by side.

In [ ]:
def run(label, config, backend=None, policy="position"):
    enc = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
    kwargs = {"config": config}
    if backend:
        kwargs["quant_backend"] = backend
    cache = create_tiered_cache(model, dtype=DTYPE, **kwargs)
    tracker = AttentionTracker(num_layers=num_layers)
    out = forward_with_tracking(model, enc.input_ids, cache, tracker=tracker)
    tok = out.logits[:, -1:, :].argmax(dim=-1)
    for step in range(MAX_NEW - 1):
        out = forward_with_tracking(model, tok, cache, tracker=tracker)
        tok = out.logits[:, -1:, :].argmax(dim=-1)
        if (step + 1) % REASSIGN_EVERY == 0:
            if policy == "adaptive" and reassign_tiers_adaptive is not None:
                reassign_tiers_adaptive(cache, tracker, num_layers)
            else:
                reassign_tiers_by_position(cache, num_layers)
        if tokenizer.eos_token_id and tok.item() == tokenizer.eos_token_id:
            break
    return cache.compressed_storage_bytes() / (1024 * 1024)

In [ ]:
# Baseline
enc = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
bc = DynamicCache()
with torch.no_grad():
    model.generate(**enc, past_key_values=bc, max_new_tokens=MAX_NEW, do_sample=False)
base = cache_storage_bytes(bc) / (1024 * 1024)
print(f"Baseline FP16: {base:.2f} MiB")

In [ ]:
safe = run("Safe", FadeConfig.safe())
rot2 = run("Rot2", FadeConfig.safe(), get_backend("rotated", head_dim=head_dim, bits=2))
bal = run("Bal", FadeConfig.balanced().with_overrides(eviction_policy="position"))
ada = run(
    "Ada",
    FadeConfig(phase="2", int4_budget=200, int2_budget=200, eviction_policy="position"),
    policy="adaptive",
)
agg = run("Agg", FadeConfig.aggressive().with_overrides(eviction_policy="position"))

In [ ]:
print(f"\n{'=' * 55}")
print(f"  FADE Results — {MODEL_ID}")
print(f"{'=' * 55}")
print(f"  {'Config':<25} {'MiB':>6} {'Ratio':>7}")
print(f"  {'-' * 25} {'-' * 6} {'-' * 7}")
for name, kv in [
    ("Baseline FP16", base),
    ("Safe INT4", safe),
    ("Rotated 2-bit", rot2),
    ("Balanced", bal),
    ("Adaptive (E1)", ada),
    ("Aggressive", agg),
]:
    print(f"  {name:<25} {kv:>5.2f} {base / kv:>6.1f}x")